# 06 — Déploiement

**Début** : charge `05_model_registry`. **Fin** : échantillons + state.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "mlops").is_dir():
        REPO_ROOT = p
        break
sys.path.insert(0, str(REPO_ROOT))

import copy
import json

from diffusers import AutoencoderKL, StableDiffusionPipeline, UNet2DConditionModel
from peft import PeftModel
from transformers import CLIPTextModel, CLIPTokenizer

from shared.paths import STEP_06_SAMPLES_DIR
from shared.pokemon_dataset import metadata_to_conditioning, pick_device
from shared.sd_lora_models import DEFAULT_MODEL_ID
from shared.step_artifacts import load_previous_step_output, resolve_path, rel_path, save_step_output

prev = load_previous_step_output("06_deployment")
lora_dir = resolve_path(prev["lora_dir"])
model_id = prev.get("model_id", DEFAULT_MODEL_ID)

device = pick_device()
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)
unet = PeftModel.from_pretrained(
    UNet2DConditionModel.from_pretrained(model_id, subfolder="unet"),
    lora_dir,
).to(device)
unet.eval()
unet_merged = copy.deepcopy(unet).merge_and_unload()
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, vae=vae, unet=unet_merged, text_encoder=text_encoder,
    tokenizer=tokenizer, safety_checker=None,
).to(device)

def generate_card_from_metadata(meta, num_inference_steps=200, guidance_scale=7.5, save_path=None):
    image = pipe(metadata_to_conditioning(meta), num_inference_steps=num_inference_steps, guidance_scale=guidance_scale).images[0]
    if save_path:
        image.save(save_path)
    return image


In [ ]:
STEP_06_SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
example_meta = {
    "name": "Charizard", "types": ["Fire"], "hp": 120, "stage": "Stage2", "rarity": "Rare",
    "attacks": [{"name": "Fire Spin", "damage": "100"}],
}
p1 = STEP_06_SAMPLES_DIR / "charizard_example.png"
generate_card_from_metadata(example_meta, save_path=p1)
sample_paths = [rel_path(p1)]

from shared.step_artifacts import load_step_output
fe = load_step_output("01_feature_engineering")
meta_path = resolve_path(fe["metadata_path"])
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    p2 = STEP_06_SAMPLES_DIR / "from_feature_engineering.png"
    generate_card_from_metadata(meta, num_inference_steps=250, save_path=p2)
    sample_paths.append(rel_path(p2))

save_step_output("06_deployment", {
    "lora_dir": prev["lora_dir"],
    "model_id": model_id,
    "deployment_samples_dir": rel_path(STEP_06_SAMPLES_DIR),
    "sample_paths": sample_paths,
})
